# Build Japanese–English speech pairs with Kokoro

Synthesize both sides of JParaCrawl text pairs with Kokoro, then check the audio with Whisper. The notebook saves progress and outputs to Drive.

| Goal | Cells to run | Runtime |
| --- | --- | --- |
| First smoke test | **1 → 2 → 3**. Review 10 pairs, then try 100. | L4 GPU |
| Tune performance and validate the settings | **4 → 5 → 6**. Test 1,000 pairs with the selected settings. | L4 GPU |
| Start or resume production | **1 → 2 → 5 → 7 → 8**. Reuse saved settings when resuming. | L4 GPU |
| Export after all shards finish | **1 → 2 → 9**. Set `RELEASE_ONLY = True` in cell 2. | CPU supported |

Run the cells for your current stage. Start at cell 1 in a fresh runtime. Reuse completed tests and resume with the same inputs, configuration, and output path on Drive.

[Kokoro runbook](https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus/blob/main/docs/kokoro-runbook.md)


## 1. Mount Drive and set up the runtime

Select **L4 GPU** for audio generation or **CPU (no hardware accelerator)** for release export only. This cell mounts Drive, fetches `main`, and sets up Python 3.12 and its dependencies.

If a previous manual patch causes an uncommitted-changes error, append a suffix such as `-v2` to `REPO_DIR` and rerun the cell. This creates a fresh checkout and reuses the saved configuration and outputs on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, json
from pathlib import Path
REPO_DIR = Path('/content/ja-en-direct-s2st-corpus-kokoro')
BRANCH = 'main'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus.git', str(REPO_DIR)], check=True)
else:
    dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True)
    assert not dirty.strip(), 'Save or commit local changes before updating the checkout.'
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', 'origin/' + BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'espeak-ng', 'build-essential', 'cmake'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'], check=True)
UV = [sys.executable, '-m', 'uv']
subprocess.run(UV + ['python', 'install', '3.12'], check=True)
ENV_DIR = Path('/content/s2st-kokoro-py312')
if not ENV_DIR.exists():
    subprocess.run(UV + ['venv', '--python', '3.12', '--seed', str(ENV_DIR)], check=True)
PYTHON = str(ENV_DIR / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', 'torch==2.7.1', 'torchaudio==2.7.1', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', 'requirements/kokoro-colab.txt'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)
# Download the pinned dictionary separately from the UniDic Python package.
dictionary_check = "from pathlib import Path; import unidic; p=Path(unidic.DICDIR); assert (p/'sys.dic').is_file() and (p/'version').read_text().strip() == 'unidic-3.1.0+2021-08-31'"
if subprocess.run([PYTHON, '-c', dictionary_check]).returncode:
    subprocess.run([PYTHON, '-m', 'unidic', 'download', '3.1.0+2021-08-31'], check=True)
# Clear Qwen batch overrides.
os.environ.pop('S2ST_TTS_BATCH_SIZE', None)
os.environ.pop('S2ST_ASR_BATCH_SIZE', None)
os.environ['HF_HOME'] = '/content/hf-kokoro-cache'
os.environ['S2ST_INPUT_INDEX_DIR'] = '/content/s2st-input-index'
os.environ['S2ST_MODEL_ROOT'] = '/content/kokoro-models'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Runtime setup is complete. Load the configuration in cell 2.', flush=True)


## 2. Load the configuration

For **release export only**, set `RELEASE_ONLY = True`, run this cell, and continue to **cell 9**. The notebook loads the saved configuration and skips GPU checks and model downloads.

For a first run or to resume generation, use `RELEASE_ONLY = False` to check the input counts and SHA-256, GPU, and free space. For a first run, export `manifest.jsonl` and `summary.json` to `jparacrawl-filtering/outputs/text-corpus/`.

On Drive, the configuration is `ja-en-direct-s2st-corpus/kokoro-jparacrawl-l4-config.yaml`. Logs are under `kokoro-logs/` in the same directory.


In [ ]:
RELEASE_ONLY = False  # Set True to export a finished release.

import shutil
DATA_ROOT = Path('/content/drive/MyDrive/ja-en-direct-s2st-corpus')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = DATA_ROOT / 'kokoro-jparacrawl-l4-config.yaml'
if not CONFIG.exists():
    if RELEASE_ONLY:
        raise FileNotFoundError(f'Production configuration not found: {CONFIG}')
    shutil.copyfile(REPO_DIR / 'configs/production-kokoro-jparacrawl-l4.yaml', CONFIG)
print(CONFIG.read_text())
# Read paths using YAML from the dedicated Python environment.
def read_settings():
    script = 'import yaml,json,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))'
    return json.loads(subprocess.check_output([PYTHON, '-c', script, str(CONFIG)], text=True))
settings = read_settings()
for key in ('manifest', 'summary'):
    path = Path(settings['prepare'][key])
    assert path.is_file(), f'Export input not found: {path}'
    print(key, path, path.stat().st_size)
# Save the installed dependencies and code revision.
ENV_REPORT = DATA_ROOT / 'kokoro-environment'
ENV_REPORT.mkdir(exist_ok=True)
(ENV_REPORT / 'pip-freeze.txt').write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True))
(ENV_REPORT / 'git-commit.txt').write_text(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True))

from datetime import datetime
LOG_DIR = DATA_ROOT / 'kokoro-logs'
LOG_DIR.mkdir(exist_ok=True)
def run_command(command, *args, production=False):
    cmd = [PYTHON, '-u', '-m', 's2st_corpus.cli', '--config', str(CONFIG)]
    if production:
        cmd.append('--allow-production')
    cmd.extend([command, *map(str, args)])
    log_path = LOG_DIR / (datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '-' + command + '.log')
    print('Log:', log_path, flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                rc = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if rc:
        raise RuntimeError(f'{command} failed ({rc}): {log_path}')

if RELEASE_ONLY:
    print('Release setup is complete. Continue to cell 9.', flush=True)
else:
    print('Validating inputs.', flush=True)
    run_command('prepare')
    print('Checking GPU and available storage.', flush=True)
    run_command('preflight')
    print('Preparing the speech model, dictionary, and voices.', flush=True)
    run_command('preload')
    print('Input and GPU checks are complete. Run the 10-pair test in cell 3.', flush=True)


## 3. Test 10 pairs, then 100

Run this cell with `SMOKE_PAIRS = 10` and review the accepted Japanese and English audio and counts. Change it to `SMOKE_PAIRS = 100` and run the same cell again. Results are stored separately by sample size. Rejection reasons are in `all.jsonl`.

Set `RUN_VOICE_SAMPLES = True` to listen to candidate voices. Select the production voices in the configuration file on Drive.


In [ ]:
SMOKE_PAIRS = 10  # Review the 10-pair result before changing this to 100.
RUN_SMOKE = True  # Set False to skip an already completed test.
RUN_VOICE_SAMPLES = False  # Enable to listen to candidate voices.

VOICE_DIR = DATA_ROOT / 'kokoro-voice-samples'
if RUN_VOICE_SAMPLES:
    run_command('voice-samples', '--output-dir', VOICE_DIR, '--en-voices', 'af_heart', 'af_bella', 'am_adam')
    from IPython.display import Audio, display
    for path in sorted(VOICE_DIR.glob('*.wav')):
        print(path.name)
        display(Audio(filename=str(path)))

if RUN_SMOKE:
    assert SMOKE_PAIRS in (10, 100), 'Choose 10 or 100 pairs here. Use cell 6 for 1,000 pairs.'
    run_command('smoke-test', '--pairs', SMOKE_PAIRS)
    settings = read_settings()
    release_dir = Path(settings['smoke']['output_dir']) / f'pairs-{SMOKE_PAIRS}/production/manifests/releases'
    summary = json.loads((release_dir / 'summary.json').read_text(encoding='utf-8'))
    print('Results directory:', release_dir)
    keys = ['total_pairs', 'accepted_pairs', 'rejected_pairs', 'acceptance_rate', 'failures', 'tts_rtf_total']
    print(json.dumps({key: summary.get(key) for key in keys}, ensure_ascii=False, indent=2))
    with (release_dir / 'accepted.jsonl').open(encoding='utf-8') as manifest:
        first_line = manifest.readline()
    if first_line:
        from IPython.display import Audio, display
        pair = json.loads(first_line)
        for lang in ('ja', 'en'):
            print(lang, pair[f'{lang}_text'])
            display(Audio(filename=pair[f'{lang}_wav_16k']))
    else:
        print('No pairs passed QC. See all.jsonl for rejection reasons.')
else:
    print('Smoke test skipped.')


## 4. Benchmark throughput on L4

After reviewing the 100-pair test, set `RUN_L4_TUNING = True`. The benchmark compares Kokoro worker counts of 1, 2, 4, and 6 and Whisper batch sizes of 8, 16, 24, and 32 on the same 200 pairs.

Use `TUNING_STAGE = 'all'` for both stages. The `'asr'` option reruns Whisper with benchmark audio from the same runtime and code commit. The report records actual sample allocation, throughput, VRAM, and quality comparisons.


In [ ]:
RUN_L4_TUNING = False  # Enable after reviewing the 100-pair test.
TUNING_STAGE = 'all'  # all runs both stages, tts runs synthesis, and asr runs transcription.

if RUN_L4_TUNING:
    assert TUNING_STAGE in ('all', 'tts', 'asr')
    run_command('preflight')
    run_command('benchmark-performance', '--stage', TUNING_STAGE)
else:
    print('Benchmark skipped. Showing the saved report if available.')

PERFORMANCE_REPORT = Path(read_settings().get('performance', {}).get('report_dir', str(DATA_ROOT / 'kokoro-performance'))) / 'l4-performance.json'
if PERFORMANCE_REPORT.is_file():
    performance_report = json.loads(PERFORMANCE_REPORT.read_text())
    print('GPU:', performance_report.get('gpu'))
    print('Recommended Kokoro workers:', performance_report.get('recommended_tts_workers'))
    print('Recommended Whisper batch size:', performance_report.get('recommended_asr_batch_size'))
    for stage in ('tts_candidates', 'asr_candidates'):
        print('Synthesis candidates:' if stage == 'tts_candidates' else 'Transcription candidates:')
        for candidate in performance_report.get(stage, []):
            keys = ['worker_count', 'batch_size', 'status', 'audio_hours_per_wall_hour', 'audio_seconds_per_wall_second', 'headroom_warning', 'quality_equivalent', 'gpu_peak_used_gib_sampled', 'peak_reserved_vram_gib_reference', 'error']
            print({key: candidate.get(key) for key in keys})
else:
    print('No saved benchmark report is available.')

print('Higher audio_hours_per_wall_hour or audio_seconds_per_wall_second means higher throughput.')
print('quality_equivalent indicates matching quality. headroom_warning flags low VRAM headroom.')
print('gpu_peak_used_gib_sampled reports the highest total GPU usage sampled every five seconds.')
print('peak_reserved_vram_gib_reference reports reference values from per-process peaks.')


## 5. Save the selected settings

Review the benchmark and set **`tts.worker_count`** and **`asr.batch_size`** in the configuration on Drive. Verify the values printed by this cell, then set `PERFORMANCE_CONFIG_CONFIRMED = True`.

When resuming production, confirm the saved settings. Use a fresh `smoke.output_dir` when testing a different voice, model, or Whisper batch size.


In [ ]:
settings = read_settings()
print('Config:', CONFIG)
print('Kokoro workers:', settings['tts'].get('worker_count', 1))
print('Whisper batch:', settings['asr']['batch_size'])
PERFORMANCE_CONFIG_CONFIRMED = False
print('Configuration:', 'Confirmed. Continue to cell 6.' if PERFORMANCE_CONFIG_CONFIRMED else 'Review the displayed settings.')


## 6. Test 1,000 pairs with the selected settings

After confirming the configuration in cell 5, set `RUN_BENCHMARK_1000 = True`. The test samples 1,000 training pairs by text length, generates audio, and runs QC. By default, it reallocates any length-bin shortfall to other bins and records the actual counts.

Review acceptance, CER/WER, long-utterance failures, and runtime. The results go to `smoke.output_dir/pairs-1000/production/manifests/releases/`.


In [ ]:
RUN_BENCHMARK_1000 = False
BENCHMARK_SHORTFALL = 'redistribute'  # Use 'error' to enforce the original length-bin quotas.
if RUN_BENCHMARK_1000:
    assert PERFORMANCE_CONFIG_CONFIRMED, 'Review the recommendations and confirm the selected settings before running.'
    run_command('smoke-test', '--pairs', 1000, '--benchmark-seed', 42,
                '--benchmark-shortfall', BENCHMARK_SHORTFALL)
    settings = read_settings()
    report_path = Path(settings['smoke']['output_dir']) / 'pairs-1000/production/manifests/releases/summary.json'
    report = json.loads(report_path.read_text())
    keys = ['total_pairs', 'length_bin_distribution', 'accepted_pairs', 'rejected_pairs', 'ja_hours', 'en_hours', 'attempt_metrics', 'failures', 'duration_failures_by_length_bin', 'duration_failures_by_language_and_length_bin', 'tts_rtf_total', 'asr_phase_wall_seconds', 'asr_inference_seconds', 'full_pipeline_wall_seconds', 'checkpoint_writes', 'checkpoint_write_seconds', 'directory_scan_count', 'directory_scan_seconds', 'train', 'dev', 'test']
    view = {key: report.get(key) for key in keys}
    view['gpu'] = [i.get('gpu') for i in report['invocations']]
    view['parent_peak_allocated_vram_gib'] = max((i.get('peak_allocated_vram_gib', 0) for i in report['invocations']), default=0)
    view['parent_peak_reserved_vram_gib'] = max((i.get('peak_reserved_vram_gib', 0) for i in report['invocations']), default=0)
    view['tts_worker_per_process_peaks'] = [i.get('tts_worker_per_process_peak', {}) for i in report['invocations']]
    view['phase_throughput'] = [{key: i.get(key) for key in ('kokoro_worker_count', 'asr_batch_size', 'tts_generated_utterances', 'tts_generated_audio_seconds', 'tts_phase_seconds', 'tts_audio_hours_per_wall_hour', 'asr_audio_seconds', 'asr_phase_seconds')} for i in report['invocations']]
    print(json.dumps(view, ensure_ascii=False, indent=2))
else:
    print('The 1,000-pair test is disabled. Confirm the configuration in cell 5 before enabling it.')


## 7. Check production progress

`completed_shards` shows the finished count, `remaining_shards` shows the remaining count, and `next_shard` identifies the next shard. **If zero shards remain, go to cell 9.**


In [ ]:
run_command('plan')
run_command('status')


## 8. Generate production audio and run QC

After reviewing the 1,000-pair test, set `APPROVE_PRODUCTION = True` to start. Begin with `RUN_ALL_REMAINING = False` to run one shard. Review the result, then set it to `True` to process the rest.

When the cell finishes and shows `completed=128/128 remaining=0`, **export the release in cell 9**. Audio and progress are stored in `run.output_dir` on Drive. Use one runtime to write to that directory.


In [ ]:
APPROVE_PRODUCTION = False  # Enable production after reviewing the 1,000-pair test.
RUN_ALL_REMAINING = False  # False runs one shard. True runs all remaining shards.

if APPROVE_PRODUCTION:
    assert PERFORMANCE_CONFIG_CONFIRMED, 'Confirm the production settings in cell 5.'
    if RUN_ALL_REMAINING:
        run_command('run-remaining-shards', '--skip-consolidate', production=True)
    else:
        run_command('run-next-shard', production=True)
else:
    print('Production is disabled. Review the 1,000-pair test before enabling it.')


## 9. Export the release (CPU supported)

After all shards finish, set `WRITE_RELEASE = True` and run the cell. The release is written under `run.output_dir/manifests/releases/`.

| Mode | Checks |
| --- | --- |
| `'inventory'` | Checks WAV file presence and QC records. Uses the recorded audio metadata and hashes. |
| `'full'` (default) | Opens each WAV and verifies its format, duration, and SHA-256 digest. |

Set `RELEASE_AUDIO_VERIFICATION = 'inventory'` for a faster export on Drive. The summary records the selected mode under `audio_verification`.

Confirm **`complete: true`** and **`missing_shards: []`** in `summary.json`. Use **`accepted.jsonl`**, which contains both train and dev, for downstream training.


In [ ]:
WRITE_RELEASE = False  # Enable after confirming that all shards finished.
RELEASE_AUDIO_VERIFICATION = 'full'  # 'inventory' checks file presence and QC records.
if WRITE_RELEASE:
    run_command('consolidate', '--audio-verification', RELEASE_AUDIO_VERIFICATION)
    release_dir = Path(read_settings()['run']['output_dir']) / 'manifests/releases'
    summary = json.loads((release_dir / 'summary.json').read_text(encoding='utf-8'))
    print('Release directory:', release_dir)
    print('Manifest for downstream training:', release_dir / 'accepted.jsonl')
    keys = ('complete', 'missing_shards', 'total_pairs', 'accepted_pairs', 'rejected_pairs', 'audio_verification')
    print({key: summary.get(key) for key in keys})
else:
    print('Release export is disabled. Test results are in their pairs-N directories.')
